### **Connect Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Import Libraries & Install Dependencies**

In [ ]:
import re
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
!pip install scikit-multilearn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 4.8 MB/s eta 0:00:00


### **Import Dataset**

In [ ]:
#Read Dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Datasets/en_hf.csv')
print(df.head())

                                                text  labels source dataset  \
0  According to a recent OFSTED report, a school ...     1.0    NGO   CONAN   
1  In Birmingham there is a school where girls an...     1.0    NGO   CONAN   
2  A school in Birningham is still separating gir...     1.0    NGO   CONAN   
3  The police and politicians are covering up the...     1.0    NGO   CONAN   
4  Muslims grooming gangs are protected by the go...     1.0    NGO   CONAN   

  nb_annotators post_author_country_location  
0             1                      unknown  
1             1                      unknown  
2             1                      unknown  
3             1                      unknown  
4             1                      unknown  


In [ ]:
print(df.columns.tolist())

['text', 'labels', 'source', 'dataset', 'nb_annotators', 'post_author_country_location']


### **Word List**

In [ ]:
SLANG_WORDS = {
  # ── GEN-Z CORE SLANG ──
  "slay", "slays", "slaying",
  "rizz", "rizzing", "rizzy", "rizzler",
  "delulu",                            # delusional (Gen-Z shortening)
  "bussin",                            # very good (AAVE/Gen-Z)
  "sheesh",
  "periodt",                           # emphatic "period"
  "nocap", "cap", "capping",           # "no cap" = no lie
  "lowkey", "highkey",                 # informal intensifiers
  "mid",                               # mediocre (Gen-Z)
  "sus",                               # suspicious (Among Us)
  "vibe", "vibes", "vibing",
  "based",                             # Gen-Z approval term
  "ratio", "ratioed",                  # Twitter: outperformed in replies
  "snatched",                          # looking great (AAVE)
  "goated",                            # GOAT-level (greatest of all time)
  "npc",                               # non-playable character (Gen-Z insult)
  "slaps",                             # sounds/tastes great
  "brat",                              # Charli XCX Gen-Z aesthetic
  "demure",                            # viral Gen-Z trend
  "girlypop",
  "pookie",                            # term of endearment (Gen-Z)
  "era",                               # "I'm in my __ era" (Gen-Z)
  "yeet",                              # exclamation / throw
  "bet",                               # affirmation ("bet!" = okay!)
  "cringe", "cringy", "cringey",       # embarrassing
  "naur",                              # Gen-Z spelling of "no"
  "sksksk",                            # Gen-Z excitement expression
  "oomf",                              # one of my followers
  "moots",                             # mutuals on social media
  "bffr",                              # be for f***ing real
  "boomer",                            # older/out-of-touch person (internet)
  "karen",                             # entitled/demanding person (internet slang)
  "chad",                              # internet culture: confident man
  "incel",                             # internet culture: involuntary celibate
  "ratioed",                           # got more replies than likes (Twitter)
  "gaslit",                            # past tense of gaslight
  "glowup",                            # transformation (glow-up)
  "copium",                            # cope + opium (internet slang)
  "sigma",                             # sigma male (internet culture)
  "mogged",                            # outperformed/shown up (Gen-Z)
  "gatekeep", "gatekeeping",
  "fax",                               # "facts" slang spelling
  "hella",
  "deadass",

  # ── DIGITAL RELATIONSHIP & SOCIAL SLANG ─────────────────────────
  "ghosting", "ghosted",
  "situationship",
  "simp", "simping",
  "pickme",                            # pick-me person
  "friendzone", "friendzoned",
  "girlboss", "girlbossing",
  "boysober",
  "benching",
  "breadcrumbing",
  "lovebombing",
  "redflag",                           # single-token form; "red flag" → SLANG_PHRASES
  "bestie",

  # ── SOCIAL MEDIA IDENTITY SLANG ──────────────────────────────────
  "toxic",                             # social media: toxic relationship/person
  "gaslight", "gaslighting",
  "overthinking",
  "doomscrolling",
  "triggered",                         # internet/social media usage
  "ick",                               # sudden repulsion (Gen-Z)

  # ── DIGITAL ACTIVITY SLANG ───────────────────────────────────────
  "shadowbanned",
  "doxing", "doxxing",
  "catfish", "catfished",
  "clout",                             # social media fame/influence
  "flexing", "flex",
  "fyp",                               # For You Page (TikTok)
  "cancel", "cancelled", "canceling", "canceled",
  "exposing",

  # ── LAUGHTER EXPRESSIONS (digital-specific) ──────────────────────
  "lmaoo", "lmaooo",                   # elongated forms (base forms in ABBREV)
  "lolll",
  "haha", "hehe", "hihi", "ahahaha",  # informal laugh markers

  # ── SLANG GREETINGS & ADDRESS ────────────────────────────────────
  "bruh", "bruv",
  "girlie", "sis",
  "aight", "ight",
  "yo",
  "mf", "mofo",
  "babe", "babes",

  # ── TOXIC INTERNET EXPRESSIONS ───────────────────────────────────
  "kys", "kms",
  "stfu",
  "snowflake",
  "unhinged", "chaotic",
}

In [ ]:
SLANG_PHRASES = {
  "no cap",
  "it's giving", "its giving",
  "hits different",
  "main character", "main character energy",
  "touch grass",
  "skill issue",
  "on god",
  "for real for real",
  "say less",
  "understood the assignment",
  "left no crumbs",
  "rent free", "living rent free",
  "the audacity",
  "sending me", "this is sending me",
  "and i oop",
  "plot twist",
  "slay queen",
  "girl math", "girl dinner",
  "character development",
  "gaslight girlboss gatekeep",
  "not me",
  "bestie behavior",
  "red flag", "green flag", "beige flag",
  "soft launch", "hard launch",
  "talking stage",
  "body count",
  "love bombing",
  "dm slide",
  "soft life",
  "girl boss",
  "pick me",
  "friend zone",
  "gassed up",
  "vibe check",
  "roman era", "very demure",
  "chronically online",
  "attention seeker",
  "brat summer",
  "caught in 4k",
  "this ain't it",
  "ate and left no crumbs",
  "i'm crying",
}

In [ ]:
ABBREV_WORDS = {
  # ── LETTER SUBSTITUTION (texting-era, unambiguous) ───────────────
  "u",     # you
  "r",     # are
  "ur",    # your / you're
  "gr8",   # great
  "l8",    # late
  "l8r",   # later
  "b4",    # before
  "h8",    # hate (leet-style)
  "m8",    # mate (leet-style)
  "sk8",   # skate (leet-style)
  "abt",   # about
  "bc",    # because
  "rn",    # right now
  "imo",   # in my opinion
  "ngl",   # not gonna lie
  "tbh",   # to be honest
  "imho",  # in my humble opinion
  "afaik", # as far as I know
  "afk",   # away from keyboard
  "irl",   # in real life
  "tldr",  # too long didn't read

  # ── TEXTING ABBREVIATIONS ────────────────────────────────────────
  "omw",   # on my way
  "otw",   # on the way
  "smh",   # shaking my head
  "istg",  # I swear to god
  "idk",   # I don't know
  "idc",   # I don't care
  "idgaf", # I don't give a f
  "wdym",  # what do you mean
  "wdyt",  # what do you think
  "wbu",   # what about you
  "hbu",   # how about you
  "fyi",   # for your information
  "btw",   # by the way
  "jk",    # just kidding
  "jkjk",
  "nvm",   # nevermind
  "np",    # no problem
  "yw",    # you're welcome
  "tysm",  # thank you so much
  "ily",   # I love you
  "ilysm", # I love you so much
  "ikr",   # I know right
  "lmk",   # let me know
  "hmu",   # hit me up
  "ttyl",  # talk to you later
  "ttys",  # talk to you soon
  "gtg",   # got to go
  "g2g",   # got to go (also detected by leet-speak regex)
  "bbl",   # be back later
  "brb",   # be right back
  "brt",   # be right there
  "omg",   # oh my god
  "wtf",   # what the f
  "wth",   # what the heck
  "rofl",  # rolling on the floor laughing
  "lol",   # laugh out loud
  "lmao",  # laughing my a off
  "lmfao",
  "asap",  # as soon as possible
  "tbt",   # throwback thursday
  "ootd",  # outfit of the day
  "grwm",  # get ready with me
  "iykyk", # if you know you know
  "fomo",  # fear of missing out
  "yolo",  # you only live once
  "goat",  # greatest of all time
  "pov",   # point of view
  "tw",    # trigger warning
  "cw",    # content warning
  "icymi", # in case you missed it
  "fwiw",  # for what it's worth
  "mfw",   # my face when
  "tfw",   # that feeling when
  "nbd",   # no big deal
  "atm",   # at the moment
  "til",   # today I learned
  "eli5",  # explain like I'm 5
  "ftfy",  # fixed that for you
  "glhf",  # good luck have fun
  "gratz", # congratulations
  "iirc",  # if I recall correctly
  "op",    # original poster
  "nsfw", "sfw",
  "gg",    # good game
  "tbf",   # to be fair
  "nts",   # note to self

  # ── PLATFORM ABBREVIATIONS ───────────────────────────────────────
  "ig",    # Instagram / "I guess"
  "tt",    # TikTok
  "yt",    # YouTube
  "fb",    # Facebook
  "twt",   # Twitter
  "dm",    # direct message
  "pm",    # private message
  "dms",

  # ── QUICK RESPONSE ABBREVIATIONS ────────────────────────────────
  "thx",   # thanks
  "tks", "tq", "ty", "tnx",
  "pls",   # please
  "plz",
  "wya",   # where you at
  "wyd",   # what you doing
  "hyd",   # how you doing
  "sup",   # what's up
  "fr",    # for real
  "ong",   # on god
  "frfr",
  "ik",    # I know

  # ── TOXIC / DISMISSIVE ABBREVIATIONS ────────────────────────────
  "idrc",  # I don't really care
  "stfu",  # shut the f up
  "gtfo",  # get the f out
  "kys",   # (toxic abbreviation)
  "kms",   # (toxic abbreviation)
  "fwb",   # friends with benefits
  "lmaooo",
}

In [ ]:
overlap = SLANG_WORDS & ABBREV_WORDS
if overlap:
    print(f"[INFO] Overlap SLANG_WORDS ∩ ABBREV_WORDS ({len(overlap)} kata): {overlap}")
    print("Overlap di atas disengaja (toxic markers yang sekaligus singkatan).")
print(f"[INFO] SLANG_WORDS  : {len(SLANG_WORDS)} kata")
print(f"[INFO] SLANG_PHRASES: {len(SLANG_PHRASES)} frasa")
print(f"[INFO] ABBREV_WORDS : {len(ABBREV_WORDS)} kata")

[INFO] Overlap SLANG_WORDS ∩ ABBREV_WORDS (4 kata): {'kms', 'stfu', 'lmaooo', 'kys'}
Overlap di atas disengaja (toxic markers yang sekaligus singkatan).
[INFO] SLANG_WORDS  : 118 kata
[INFO] SLANG_PHRASES: 51 frasa
[INFO] ABBREV_WORDS : 118 kata


### **Minimal Clean**

In [ ]:
def minimal_clean(text):
    if pd.isna(text):
        return ""
    t = str(text).strip()
    # Curly quotes & common Unicode artifacts
    t = (
        t.replace("\u2018", "'")
         .replace("\u2019", "'")
         .replace("\u201c", '"')
         .replace("\u201d", '"')
    )
    t = t.replace("&lt;",   " ") \
     .replace("&gt;",   " ") \
     .replace("&nbsp;", " ") \
     .replace("&apos;", "'") \
     .replace("&quot;", '"')
    # HTML entities
    t = re.sub(r"&#\d+;", " ", t)
    # Latin-1 / Windows-1252 artifacts
    t = re.sub(r"[ÃÂâ][^\s]*", " ", t)
    t = re.sub(r"\\x[0-9a-fA-F]{2}", "", t)
    # Whitespace normalization
    t = re.sub(r"\s+", " ", t).strip()
    return t

### **Detect Slang and Abbrev**

In [ ]:
def detect_slang(text: str) -> int:
    text_lower = str(text).lower()

    # (a) Single-token whole-word matching
    tokens = set(re.findall(r'\b\w+\b', text_lower))
    if tokens & SLANG_WORDS:
        return 1

    # (b) Multi-word phrase substring matching
    for phrase in SLANG_PHRASES:
        if phrase in text_lower:
            return 1

    return 0

def detect_abbrev(text: str) -> int:
    text_lower = str(text).lower()
    tokens = set(re.findall(r'\b\w+\b', text_lower))

    # (a) Kamus abbrev (whole-word match)
    if tokens & ABBREV_WORDS:
        return 1

    # (b) Leet-speak: huruf + digit + huruf (gr8, l8r, b4, g2g)
    if re.search(r'\b(?:[a-z]+[0-9][a-z0-9]*|[0-9]+[a-z][a-z0-9]*)\b', text_lower):
        return 1

    return 0

### **Multi-Dimensional Stratified Sampling**

In [ ]:
def _multi_stratified_sample(df: pd.DataFrame, n_per_class: int, seed: int) -> pd.DataFrame:
    groups = []

    for lbl in sorted(df["label"].unique()):
        subset = df[df["label"] == lbl].copy()
        n = min(n_per_class, len(subset))
        if n < n_per_class:
            print(f"  [!] WARN: label={lbl} hanya {len(subset):,} baris tersedia, "
                  f"menggunakan semua.")

        sampled = None

        # ── Coba skmultilearn (Iterative Stratification) ──────────────
        try:
            from skmultilearn.model_selection import iterative_train_test_split

            X = np.arange(len(subset)).reshape(-1, 1)
            y = subset[["has_slang", "has_abbrev"]].values.astype(float)

            if n >= len(subset):
                sampled = subset
            else:
                test_ratio = n / len(subset)
                # iterative_train_test_split returns (X_train, y_train, X_test, y_test)
                _, _, X_sel, _ = iterative_train_test_split(X, y, test_size=test_ratio)
                selected_idx = set(X_sel.flatten())
                sampled = subset.iloc[sorted(selected_idx)]

                shortfall = n - len(sampled)
                if 0 < shortfall:
                    remaining = subset.iloc[
                        [i for i in range(len(subset)) if i not in selected_idx]
                    ]
                    extra = remaining.sample(
                        n=min(shortfall, len(remaining)),
                        random_state=seed
                    )
                    sampled = pd.concat([sampled, extra]).reset_index(drop=True)
                    print(f"  [top-up] +{len(extra)} baris untuk tutup shortfall label={lbl}")

        except Exception as e:
            print(f"  [!] skmultilearn unavailable ({e}); fallback ke composite stratum.")

        # ── Fallback: composite stratum via sklearn ─────────────────────
        if sampled is None:
            subset["_strat"] = (
                subset["has_slang"].astype(str) + "_" + subset["has_abbrev"].astype(str)
            )
            strat_col  = subset["_strat"]
            counts     = strat_col.value_counts()
            can_strat  = (counts >= 2).all() and n < len(subset)

            if can_strat:
                try:
                    from sklearn.model_selection import train_test_split as sk_split
                    _, sampled = sk_split(
                        subset, test_size=n, stratify=strat_col, random_state=seed
                    )
                except Exception:
                    sampled = subset.sample(n=n, random_state=seed)
            else:
                sampled = subset.sample(n=n, random_state=seed)

            sampled = sampled.drop(columns=["_strat"], errors="ignore")

        groups.append(sampled)

    result = pd.concat(groups).sample(frac=1, random_state=seed).reset_index(drop=True)
    return result

### **Full Cleaning**

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    t = str(text).strip().lower()

    # Encoding fix
    t = re.sub(r"\\[ntr]", " ", t)
    t = re.sub(r"(\\\s*)+", " ", t)
    t = t.replace("&amp;", " and ")

    # Elongation: maks 2 karakter berulang (anjiiir → anjir)
    t = re.sub(r"([^.!?])\1{2,}", r"\1\1", t)

    # Sisa encoding noise
    t = re.sub(r"\bx[0-9]{2,3}\b", " ", t)
    t = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", t)

    # Twitter/X artifacts: RT
    t = re.sub(r"(?i)^rt\s+", "", t)
    t = re.sub(r"(?i)\brt\b", " ", t)
    t = re.sub(r"(?i)\bretweeted\b.*?\buser\b\)?\s*[:;]*", " ", t)

    t = re.sub(r"@\w+", " user ", t)

    # Hapus URL
    t = re.sub(r"http\S+|www\.\S+|t\.co/\S+", " ", t)

    # Hashtag: hapus #, pertahankan teks
    t = re.sub(r"#(\w+)", r"\1", t)

    # Hapus emoji & karakter di luar BMP
    t = re.sub(r"[\U00010000-\U0010ffff]", " ", t)

    # Hapus angka berdiri sendiri (bukan bagian leet-speak)
    # Hapus angka berdiri sendiri (bukan bagian leet-speak)
    # '4nj1ng': 4 diikuti huruf -> TIDAK dihapus
    # 'meet at 4': 4 diapit spasi -> DIHAPUS
    t = re.sub(r"(?<![a-z])\d+(?![a-z])", " ", t)

    # Tanda petik berlebih
    t = t.replace('"', " ")
    t = re.sub(r"(?<![a-zA-Z])'", " ", t)
    t = re.sub(r"'(?![a-zA-Z])", " ", t)

    # Tanda baca berulang
    t = re.sub(r"\?{2,}", " ?", t)
    t = re.sub(r"!{2,}", " !", t)
    t = re.sub(r"\.{4,}", "...", t)
    t = re.sub(r"[:;]{2,}", " ", t)

    # Deduplikasi 'user user' → 'user'
    t = re.sub(r"(?:\buser\b\s*){2,}", "user ", t)

    # Hapus karakter non-alfanumerik kecuali yang relevan
    # Angka dipertahankan agar leet-speak tidak rusak
    t = re.sub(r"[^a-zA-Z0-9\s',\.?!]", " ", t)

    # Sisa apostrof gantung
    t = re.sub(r"(?<![a-zA-Z])'", " ", t)
    t = re.sub(r"'(?![a-zA-Z])", " ", t)

    # Normalize whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

### **Pipeline**

In [ ]:
def run_pipeline_english(
    input_path: str,
    output_path: str,
    text_col: str    = "text",
    label_col: str   = "labels",
    source: str      = "tonneau_english",
    n_per_class: int = 6000,
    seed: int        = 42
):
    LANGUAGE      = "en"
    BUFFER_RATIO  = 1.15       # 15% buffer mengkompensasi data loss saat cleaning
    DRIFT_THRESH  = 0.05       # threshold: pergeseran distribusi > 5%

    SEP_MAIN = "=" * 62
    SEP_SUB  = "─" * 55

    print(f"\n{SEP_MAIN}")
    print(f"  PIPELINE: BAHASA INGGRIS | target akhir: {n_per_class * 2:,} baris")
    print(f"{SEP_MAIN}")

    # ────────────────────────────────────────────────────────────────────
    # [1] LOAD RAW DATA
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[1] LOAD RAW DATA")
    df = pd.read_csv(input_path)
    before_na = len(df)
    print(f"    File            : {input_path}")
    print(f"    Baris dimuat    : {before_na:>7,}")
    print(f"    Kolom           : {list(df.columns)}")

    df = df[[text_col, label_col]].copy()
    df = df.dropna(subset=[text_col, label_col])
    print(f"    Hapus missing   : {before_na - len(df)}")

    df = df.rename(columns={text_col: "text", label_col: "label"})
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    before_nb = len(df)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])].reset_index(drop=True)
    print(f"    Hapus non-binary: {before_nb - len(df)}")

    df["text"] = df["text"].astype(str).str.strip()
    before_empty = len(df)
    df = df[df["text"] != ""].reset_index(drop=True)
    print(f"    Hapus kosong    : {before_empty - len(df)}")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")
    print(f"    Total valid     : {len(df):,} baris")
    n_after_load = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)")
    tqdm.pandas(desc="    minimal_clean  ")
    df["_text_mc"] = df["text"].progress_apply(minimal_clean)
    print(f"    Dilakukan pada  : {len(df):,} baris")
    print(f"    Contoh output   : {repr(df['_text_mc'].iloc[0][:80])}")
    n_after_minimal = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [3] FEATURE DETECTION (has_slang | has_abbrev)
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[3] FEATURE DETECTION (has_slang | has_abbrev)")
    tqdm.pandas(desc="    detect_slang   ")
    df["has_slang"]  = df["_text_mc"].progress_apply(detect_slang)
    tqdm.pandas(desc="    detect_abbrev  ")
    df["has_abbrev"] = df["_text_mc"].progress_apply(detect_abbrev)

    n_sl = df["has_slang"].sum()
    n_ab = df["has_abbrev"].sum()
    n11  = ((df["has_slang"]==1) & (df["has_abbrev"]==1)).sum()
    n10  = ((df["has_slang"]==1) & (df["has_abbrev"]==0)).sum()
    n01  = ((df["has_slang"]==0) & (df["has_abbrev"]==1)).sum()
    n00  = ((df["has_slang"]==0) & (df["has_abbrev"]==0)).sum()
    print(f"    has_slang       : {n_sl:,} ({df['has_slang'].mean()*100:.1f}%)")
    print(f"    has_abbrev      : {n_ab:,} ({df['has_abbrev'].mean()*100:.1f}%)")
    print(f"    (slang, abbrev) : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    if df["has_slang"].mean() > 0.95:
        print("    [!] PERINGATAN: has_slang rate > 95% — review SLANG_WORDS (kemungkinan false positive).")
    n_after_feature = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [4] DISTRIBUTION ANALYSIS (pre-sampling, dicatat untuk Bab 3)
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[4] DISTRIBUTION ANALYSIS (pre-sampling — dicatat untuk Bab 3)")
    print(f"    {SEP_SUB}")
    for lbl_val, lbl_name in [(0, "NON-HATE"), (1, "HATE   ")]:
        sub  = df[df["label"] == lbl_val]
        n_s  = sub["has_slang"].sum()
        n_a  = sub["has_abbrev"].sum()
        n_11 = ((sub["has_slang"]==1) & (sub["has_abbrev"]==1)).sum()
        n_00 = ((sub["has_slang"]==0) & (sub["has_abbrev"]==0)).sum()
        pct  = max(len(sub), 1)
        print(f"    {lbl_name} (n={len(sub):6,}): "
              f"slang={n_s}({n_s/pct*100:.1f}%)  "
              f"abbrev={n_a}({n_a/pct*100:.1f}%)  "
              f"(1,1)={n_11}  (0,0)={n_00}")
    print(f"    {SEP_SUB}")
    pre_label_dist  = df["label"].value_counts(normalize=True).sort_index().to_dict()
    pre_slang_rate  = df["has_slang"].mean()
    pre_abbrev_rate = df["has_abbrev"].mean()
    print(f"    Proporsi label  : {pre_label_dist}")
    print(f"    Rate has_slang  : {pre_slang_rate:.4f}")
    print(f"    Rate has_abbrev : {pre_abbrev_rate:.4f}")

    # ────────────────────────────────────────────────────────────────────
    # [5] MULTI-DIMENSIONAL STRATIFIED SAMPLING
    #     Strata: label × has_slang × has_abbrev
    #     Buffer: ambil n_buffer/kelas > target → trim ke tepat n_per_class setelah cleaning
    # ────────────────────────────────────────────────────────────────────
    n_buffer = int(round(n_per_class * BUFFER_RATIO))
    print(f"\n[5] MULTI-DIMENSIONAL STRATIFIED SAMPLING")
    print(f"    Strata  : label × has_slang × has_abbrev")
    print(f"    Metode  : Iterative Stratification (Sechidis et al., 2011)")
    print(f"    Target  : {n_per_class:,}/kelas = {n_per_class*2:,} total")
    print(f"    Buffer  : {n_buffer:,}/kelas (BUFFER_RATIO={BUFFER_RATIO}) → akan di-trim setelah cleaning")
    before_samp = len(df)
    df = _multi_stratified_sample(df, n_buffer, seed)
    print(f"    {before_samp:,} → {len(df):,} baris (sampling dengan buffer)")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")
    n11 = ((df["has_slang"]==1) & (df["has_abbrev"]==1)).sum()
    n10 = ((df["has_slang"]==1) & (df["has_abbrev"]==0)).sum()
    n01 = ((df["has_slang"]==0) & (df["has_abbrev"]==1)).sum()
    n00 = ((df["has_slang"]==0) & (df["has_abbrev"]==0)).sum()
    print(f"    (slang, abbrev) : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    n_after_sampling = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [6] FULL DATA CLEANING
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[6] FULL DATA CLEANING")

    # [6a] Dedup teks mentah
    before_dup = len(df)
    df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
    n_after_dedup_raw = len(df)
    print(f"    [6a] Dedup teks mentah        : {before_dup:,} → {n_after_dedup_raw:,} "
          f"(hapus {before_dup - n_after_dedup_raw})")

    # [6b] clean_text (pada _text_mc)
    print("    [6b] Cleaning teks...")
    tqdm.pandas(desc="         clean_text     ")
    df["_text_clean"] = df["_text_mc"].progress_apply(clean_text)
    n_after_clean = len(df)
    print(f"         Selesai               : {n_after_clean:,} baris")

    # [6c] Post-cleaning filter (teks kosong + media-only + dedup cleaned)
    media_only_pat = r"(?i)\bmedia\s*only\b.*\bno\s*text\b"
    before_postcl = len(df)
    df = df[df["_text_clean"].str.strip().ne("")]
    df = df[~df["_text_clean"].str.contains(media_only_pat, regex=True, na=False)]
    df = df[df["_text_clean"].str.contains(r"[a-zA-Z0-9]", regex=True, na=False)]   # ← pengganti semantik
    df = df.drop_duplicates(subset=["_text_clean"], keep="first").reset_index(drop=True)
    n_after_postcl = len(df)
    print(f"    [6c] Filter kosong + dedup   : {before_postcl:,} → {n_after_postcl:,} "
          f"(hapus {before_postcl - n_after_postcl})")

    # [6d] Filter panjang token — outlier removal (< 3 atau > 512 token)
    df["_token_len"] = df["_text_clean"].apply(lambda t: len(t.split()))
    before_len = len(df)
    df = df[(df["_token_len"] >= 3) & (df["_token_len"] <= 512)]
    df = df.drop(columns=["_token_len"]).reset_index(drop=True)
    n_after_lenfilter = len(df)
    print(f"    [6d] Filter panjang (3–512)  : {before_len:,} → {n_after_lenfilter:,} "
          f"(hapus {before_len - n_after_lenfilter})")

    # ────────────────────────────────────────────────────────────────────
    # [7] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[7] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI")

    # Shuffle sebelum trim agar seleksi akhir bersifat random
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Trim ke tepat n_per_class per label → total persis n_per_class * 2
    trimmed = []
    for lbl in [0, 1]:
        group = df[df["label"] == lbl]
        if len(group) >= n_per_class:
            trimmed.append(group.head(n_per_class))
        else:
            print(f"    [!] WARNING: label={lbl} hanya {len(group):,} baris "
                  f"(target: {n_per_class:,}) — menggunakan semua yang tersedia.")
            trimmed.append(group)
    df = pd.concat(trimmed).sample(frac=1, random_state=seed).reset_index(drop=True)
    n_final = len(df)
    print(f"    Trim ke target  : {n_after_lenfilter:,} → {n_final:,} baris")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")

    # Verifikasi drift distribusi pre-sampling vs post-cleaning
    post_label_dist  = df["label"].value_counts(normalize=True).sort_index().to_dict()
    post_slang_rate  = df["has_slang"].mean()
    post_abbrev_rate = df["has_abbrev"].mean()
    print(f"\n    Verifikasi drift distribusi (threshold Δ > {DRIFT_THRESH:.0%})")
    print(f"    {SEP_SUB}")
    drifted = False
    for lbl_val in [0, 1]:
        pre_v  = pre_label_dist.get(lbl_val, 0)
        post_v = post_label_dist.get(lbl_val, 0)
        delta  = abs(post_v - pre_v)
        flag   = " [!] DRIFT" if delta > DRIFT_THRESH else " ✓"
        print(f"    Label={lbl_val}: pre={pre_v:.4f} → post={post_v:.4f}  Δ={delta:.4f}{flag}")
        if delta > DRIFT_THRESH:
            drifted = True
    s_delta = abs(post_slang_rate  - pre_slang_rate)
    a_delta = abs(post_abbrev_rate - pre_abbrev_rate)
    flag_s  = " [!] DRIFT" if s_delta > DRIFT_THRESH else " ✓"
    flag_a  = " [!] DRIFT" if a_delta > DRIFT_THRESH else " ✓"
    print(f"    has_slang : pre={pre_slang_rate:.4f} → post={post_slang_rate:.4f}  Δ={s_delta:.4f}{flag_s}")
    print(f"    has_abbrev: pre={pre_abbrev_rate:.4f} → post={post_abbrev_rate:.4f}  Δ={a_delta:.4f}{flag_a}")
    if s_delta > DRIFT_THRESH: drifted = True
    if a_delta > DRIFT_THRESH: drifted = True
    print(f"    {SEP_SUB}")
    if drifted:
        print("    [!] Distribusi bergeser signifikan. Pertimbangkan re-sampling atau")
        print("        dokumentasikan sebagai limitasi di Bab 3.")
    else:
        print("    ✓  Tidak ada drift signifikan. Distribusi terjaga dengan baik.")

    # Tabel transisi data (wajib dilaporkan di Bab 3)
    print(f"\n    TABEL TRANSISI DATA (Bab 3 — Tabel transisi_cleaning)")
    print(f"    {SEP_SUB}")
    print(f"    [1] Data mentah (raw CSV)                       : {before_na:>7,}")
    print(f"    [2] Setelah validasi (NaN / kosong / non-binary): {n_after_load:>7,}")
    print(f"    [3] Setelah minimal cleaning                    : {n_after_minimal:>7,}")
    print(f"    [4] Setelah feature detection                   : {n_after_feature:>7,}")
    print(f"    [5] Setelah stratified sampling (buffer ×{BUFFER_RATIO})   : {n_after_sampling:>7,}")
    print(f"    [6a] Setelah dedup teks mentah                  : {n_after_dedup_raw:>7,}")
    print(f"    [6b] Setelah full cleaning                       : {n_after_clean:>7,}")
    print(f"    [6c] Setelah post-cleaning filter               : {n_after_postcl:>7,}")
    print(f"    [6d] Setelah filter panjang token (3–512)       : {n_after_lenfilter:>7,}")
    print(f"    [7]  Setelah trim ke target                     : {n_final:>7,}")
    print(f"    {SEP_SUB}")
    print(f"    FINAL DATASET                                   : {n_final:>7,}")

    # ────────────────────────────────────────────────────────────────────
    # [8] SAVE
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[8] SAVE")

    # Assign ID sequential mulai dari 1 (bukan UUID)
    df["id"]       = range(1, len(df) + 1)
    df["language"] = LANGUAGE
    df["source"]   = source

    # [8b] Final CSV — tepat 7 kolom sesuai skema Bab 3
    df_final = pd.DataFrame({
        "id"        : df["id"],
        "text"      : df["_text_clean"],  # bloom-cleaned, lowercased, siap BLOOM
        "label"     : df["label"],
        "language"  : df["language"],
        "has_slang" : df["has_slang"],
        "has_abbrev": df["has_abbrev"],
        "source"    : df["source"],
    })
    assert list(df_final.columns) == [
        "id", "text", "label", "language", "has_slang", "has_abbrev", "source"
    ], f"[ERROR] Kolom tidak sesuai: {list(df_final.columns)}"

    df_final.to_csv(output_path, index=False)
    print(f"    [8b] Final CSV   : {output_path}")
    print(f"         Kolom       : {list(df_final.columns)}")

    # Ringkasan akhir
    n11 = ((df_final["has_slang"]==1) & (df_final["has_abbrev"]==1)).sum()
    n10 = ((df_final["has_slang"]==1) & (df_final["has_abbrev"]==0)).sum()
    n01 = ((df_final["has_slang"]==0) & (df_final["has_abbrev"]==1)).sum()
    n00 = ((df_final["has_slang"]==0) & (df_final["has_abbrev"]==0)).sum()
    print(f"\n{SEP_MAIN}")
    print(f"  RINGKASAN AKHIR — BAHASA INGGRIS")
    print(f"{SEP_MAIN}")
    print(f"  Total baris      : {len(df_final):,}")
    print(f"  Distribusi label : {df_final['label'].value_counts().sort_index().to_dict()}")
    print(f"  has_slang        : {df_final['has_slang'].sum():,} ({df_final['has_slang'].mean()*100:.1f}%)")
    print(f"  has_abbrev       : {df_final['has_abbrev'].sum():,} ({df_final['has_abbrev'].mean()*100:.1f}%)")
    print(f"  (slang, abbrev)  : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    print(f"  ID range         : {df_final['id'].iloc[0]} – {df_final['id'].iloc[-1]}")
    print(f"  Kolom output     : {list(df_final.columns)}")
    print(f"  Contoh 5 baris pertama:")
    print(df_final[["id", "text", "label", "has_slang", "has_abbrev"]].head().to_string())
    print(f"{SEP_MAIN}")

    return df_final

### **Run Code**

In [ ]:
df_en = run_pipeline_english(
    input_path  = '/content/drive/MyDrive/Colab Datasets/en_hf.csv',
    output_path = '/content/drive/MyDrive/Colab Datasets/dataset_english_final.csv',
    text_col    = 'text',
    label_col   = 'labels',
    source      = 'tonneau_english',
    n_per_class = 6000,   # 6.000 hate + 6.000 non-hate = 12.000 total EN
    seed        = 42
)


  PIPELINE: BAHASA INGGRIS | target akhir: 12,000 baris

[1] LOAD RAW DATA
    File            : /content/drive/MyDrive/Colab Datasets/en_hf.csv
    Baris dimuat    : 360,493
    Kolom           : ['text', 'labels', 'source', 'dataset', 'nb_annotators', 'post_author_country_location']
    Hapus missing   : 1
    Hapus non-binary: 0
    Hapus kosong    : 23
    Distribusi label: {0: 262992, 1: 97477}
    Total valid     : 360,469 baris

[2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)


    minimal_clean  : 100%|██████████| 360469/360469 [00:12<00:00, 28069.01it/s]


    Dilakukan pada  : 360,469 baris
    Contoh output   : 'According to a recent OFSTED report, a school in Birmingham is still segregating'

[3] FEATURE DETECTION (has_slang | has_abbrev)


    detect_abbrev  : 100%|██████████| 360469/360469 [00:08<00:00, 44529.56it/s]


    has_slang       : 13,554 (3.8%)
    has_abbrev      : 48,642 (13.5%)
    (slang, abbrev) : (1,1)=3032  (1,0)=10522  (0,1)=45610  (0,0)=301305

[4] DISTRIBUTION ANALYSIS (pre-sampling — dicatat untuk Bab 3)
    ───────────────────────────────────────────────────────
    NON-HATE (n=262,992): slang=9683(3.7%)  abbrev=36381(13.8%)  (1,1)=2076  (0,0)=219004
    HATE    (n=97,477): slang=3871(4.0%)  abbrev=12261(12.6%)  (1,1)=956  (0,0)=82301
    ───────────────────────────────────────────────────────
    Proporsi label  : {0: 0.7295828490105946, 1: 0.27041715098940544}
    Rate has_slang  : 0.0376
    Rate has_abbrev : 0.1349

[5] MULTI-DIMENSIONAL STRATIFIED SAMPLING
    Strata  : label × has_slang × has_abbrev
    Metode  : Iterative Stratification (Sechidis et al., 2011)
    Target  : 6,000/kelas = 12,000 total
    Buffer  : 6,900/kelas (BUFFER_RATIO=1.15) → akan di-trim setelah cleaning
    360,469 → 13,800 baris (sampling dengan buffer)
    Distribusi label: {0: 6900, 1: 6900}
   

         clean_text     : 100%|██████████| 13794/13794 [00:01<00:00, 10205.48it/s]


         Selesai               : 13,794 baris
    [6c] Filter kosong + dedup   : 13,794 → 13,785 (hapus 9)
    [6d] Filter panjang (3–512)  : 13,785 → 13,718 (hapus 67)

[7] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI
    Trim ke target  : 13,718 → 12,000 baris
    Distribusi label: {0: 6000, 1: 6000}

    Verifikasi drift distribusi (threshold Δ > 5%)
    ───────────────────────────────────────────────────────
    Label=0: pre=0.7296 → post=0.5000  Δ=0.2296 [!] DRIFT
    Label=1: pre=0.2704 → post=0.5000  Δ=0.2296 [!] DRIFT
    has_slang : pre=0.0376 → post=0.0372  Δ=0.0004 ✓
    has_abbrev: pre=0.1349 → post=0.1289  Δ=0.0060 ✓
    ───────────────────────────────────────────────────────
    [!] Distribusi bergeser signifikan. Pertimbangkan re-sampling atau
        dokumentasikan sebagai limitasi di Bab 3.

    TABEL TRANSISI DATA (Bab 3 — Tabel transisi_cleaning)
    ───────────────────────────────────────────────────────
    [1] Data mentah (raw CSV)                       : 360,493
